<a href="https://colab.research.google.com/github/Elamathi995/Agent-calculator/blob/main/Agent_calcualtor_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# MULTI-STEP REACT AGENT WITH LOOP (NO LLM)
# ==========================================

import re

# -------------------------------
# 1. Calculator Tool (SAFE)
# -------------------------------
def calculator(expression: str) -> str:
    try:
        # Allow digits, operators, parentheses, spaces
        if not re.match(r'^[0-9+\-*/(). %*]+$', expression):
            return "Invalid expression"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# -------------------------------
# 2. Helpers
# -------------------------------
def extract_expression(text: str):
    parts = re.findall(r"[0-9\.\+\-\*\/\%\(\)\^]+", text)
    if not parts:
        return None
    expr = "".join(parts)
    expr = expr.replace("^", "**")  # support power
    return expr

def square_expression(expr: str):
    # Wrap expression safely: (expr)**2
    return f"({expr})**2"


# -------------------------------
# 3. ReAct Agent (WITH LOOP)
# -------------------------------
def react_agent(user_input: str, max_steps: int = 5):
    print("\n===== AGENT START =====")
    print("User:", user_input)

    state = {
        "current_expr": None,
        "last_result": None,
        "goal": None,  # e.g., "square_after_calc"
        "done": False
    }

    # Detect intent (very simple rules)
    text = user_input.lower()
    if "square" in text:
        state["goal"] = "square_after_calc"

    # Initial extraction
    expr = extract_expression(user_input)
    if expr:
        state["current_expr"] = expr

    # -------------------------------
    # LOOP (ReAct style)
    # -------------------------------
    for step in range(1, max_steps + 1):
        print(f"\n--- Step {step} ---")

        # Thought
        if state["done"]:
            print("Thought: I have the final answer.")
            break

        if state["current_expr"] and state["last_result"] is None:
            print("Thought: I should calculate the expression.")
            action = "calculator"
            action_input = state["current_expr"]

        elif state["goal"] == "square_after_calc" and state["last_result"] is not None:
            print("Thought: I should square the previous result.")
            action = "calculator"
            action_input = square_expression(state["last_result"])

            # Clear goal after using it once
            state["goal"] = None

        else:
            print("Thought: No further actions needed.")
            state["done"] = True
            break

        # Action
        print("Action:", action)
        print("Action Input:", action_input)

        # Observation
        observation = calculator(action_input)
        print("Observation:", observation)

        # Update state
        state["last_result"] = observation

        # Decide if done
        if state["goal"] is None:
            # No more planned steps → finish
            state["done"] = True

    # Final Answer
    if state["last_result"] is not None:
        print("\nFinal Answer:", state["last_result"])
    else:
        print("\nFinal Answer: I can only solve mathematical expressions.")

    print("===== AGENT END =====\n")


# -------------------------------
# 4. DEMO TESTS
# -------------------------------
tests = [
    "What is 25 * 4 + 10?",
    "Calculate 100 / (5 + 5)",
    "Find 7 * 8 - 3",
    "Square 5 + 5",           # multi-step: (5+5)=10 → 10^2=100
    "square 12*2",            # multi-step
    "Hello agent"
]

print("Running demo...\n")
for t in tests:
    react_agent(t)


# -------------------------------
# 5. INTERACTIVE MODE (OPTIONAL)
# -------------------------------
print("Type your query (type 'exit' to stop):\n")

while True:
    q = input("You: ")
    if q.lower() == "exit":
        print("Exiting agent ✅")
        break
    react_agent(q)

Running demo...


===== AGENT START =====
User: What is 25 * 4 + 10?

--- Step 1 ---
Thought: I should calculate the expression.
Action: calculator
Action Input: 25*4+10
Observation: 110

--- Step 2 ---
Thought: I have the final answer.

Final Answer: 110
===== AGENT END =====


===== AGENT START =====
User: Calculate 100 / (5 + 5)

--- Step 1 ---
Thought: I should calculate the expression.
Action: calculator
Action Input: 100/(5+5)
Observation: 10.0

--- Step 2 ---
Thought: I have the final answer.

Final Answer: 10.0
===== AGENT END =====


===== AGENT START =====
User: Find 7 * 8 - 3

--- Step 1 ---
Thought: I should calculate the expression.
Action: calculator
Action Input: 7*8-3
Observation: 53

--- Step 2 ---
Thought: I have the final answer.

Final Answer: 53
===== AGENT END =====


===== AGENT START =====
User: Square 5 + 5

--- Step 1 ---
Thought: I should calculate the expression.
Action: calculator
Action Input: 5+5
Observation: 10

--- Step 2 ---
Thought: I should square th